#### **1. ¿Qué son los DABs?**

Es un framework basado en Infraestructura como Código (IaC). En lugar de ir a la interfaz de Databricks y crear un Job o un Pipeline a mano (haciendo clics), escribes un archivo de configuración (YAML) que describe todo lo que necesitas. Luego, con un comando de terminal (`databricks bundle deploy`), Databricks construye todo por ti.

#### **2. Estructura de un Asset Bundle**

La guía del examen menciona específicamente que debes identificar su estructura. Un bundle típico se compone de:

- **Archivo `databricks.yml`:** Es el corazón del bundle. Aquí defines el nombre del proyecto y las configuraciones globales.

- **Recursos (Resources):** Aquí defines qué vas a desplegar, como por ejemplo:

- **Jobs:** La configuración de tus tareas programadas.

- **Delta Live Tables (DLT):** Tus pipelines declarativos.

- **Targets (Entornos):** Permite definir diferentes configuraciones para dev (desarrollo), staging y prod (producción). Por ejemplo, en "dev" puedes usar un cluster pequeño y en "prod" uno con auto-escalado.

- **Archivos de código:** Tus notebooks (`.py`, `.sql`) o archivos de Python (`.py`) que contienen la lógica del negocio.

#### **3. Diferencia con métodos tradicionales**

| Característica               | Método Tradicional (Manual/UI)                                  | Databricks Asset Bundles (DABs)                                                                 |
|-------------------------------|-----------------------------------------------------------------|-------------------------------------------------------------------------------------------------|
| Consistencia                  | Difícil de replicar exactamente en otro entorno.                | Idéntico en cada despliegue gracias al código.                                                  |
| Control de Versiones          | Los cambios en la UI no se guardan en Git fácilmente.           | Todo el pipeline vive en Git, permitiendo ver quién cambió qué.                                 |
| CI/CD                         | Requiere procesos manuales o scripts complejos.                 | Se integra nativamente con herramientas como Azure DevOps, GitHub Actions o GitLab.             |
| Escalabilidad                 | Tedioso de manejar si tienes cientos de Jobs.                   | Puedes gestionar proyectos masivos con un solo archivo de configuración.                        |

#### **4. ¿Cómo funciona en la práctica?**

Para tu meta de trabajar como Data Engineer, este será tu flujo diario:

- Desarrollas tu lógica en un IDE (como VS Code) o en Databricks Connect.

- Defines la infraestructura en el archivo YAML.

- Validas que no haya errores de sintaxis con `databricks bundle validate`.

- Despliegas a producción con un solo comando.

### **Práctica**

Imagina que queremos automatizar el proceso de ingesta de clientes que estuvimos practicando.

#### **1. Estructura de archivos del Bundle**

Esta es la organización que verías en tu computadora (por ejemplo, usando VS Code):

In [0]:
%%bash
mi_proyecto_bundle/
├── databricks.yml          # Configuración principal del bundle
├── bundle.dlt.yml          # (Opcional) Definición de pipelines de DLT
├── src/
│   └── ingesta_clientes.py # Tu código de lógica (Notebook o Python)
└── resources/
    └── jobs_config.yml     # Definición de los Jobs (tareas programadas)

#### **2. El archivo de configuración (`databricks.yml`)**

Este archivo le dice a Databricks quién eres y a dónde quieres enviar el código. Es el corazón del "Asset Bundle".

In [0]:
%%bash
bundle:
  name: ingesta-clientes-bundle

# Definimos los entornos (Targets)
targets:
  dev:
    workspace:
      host: https://adb-12345.azuredatabricks.net
    mode: development # Permite despliegues rápidos para pruebas

  prod:
    workspace:
      host: https://adb-12345.azuredatabricks.net
    mode: production # Aplica restricciones de seguridad y logs

#### **3. Definición de Recursos (`resources/jobs_config.yml`)**

Aquí es donde reemplazas los "clics" de la interfaz por código. Defines el Job que ejecutará tu tarea.

In [0]:
%%bash
resources:
  jobs:
    job_ingesta_diaria:
      name: Job_Ingesta_Clientes_Diario
      tasks:
        - task_key: procesar_bronce
          notebook_task:
            notebook_path: ../src/ingesta_clientes.py
          new_cluster: # El bundle crea el cluster automáticamente
            spark_version: "15.4.x-scala2.12"
            node_type_id: "Standard_DS3_v2"
            num_workers: 1

#### **4. Código de lógica (`src/ingesta_clientes.py`)**

Este es el archivo que ya sabes escribir, el que usa Auto Loader .

In [0]:
# logic en src/ingesta_clientes.py
import dlt

@dlt.table
def customers_raw():
  return (
    spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "parquet")
      .load("/Volumes/associate_data_engineer/default/raw_data/customers/")
  )

#### **5. ¿Cómo se despliega? (DAB vs Tradicional)**

En el examen te preguntarán por la diferencia entre este método y el tradicional.

- **Método Tradicional:** Tendrías que subir el archivo manualmente, ir a la pestaña "Workflows", crear el Job, configurar el cluster a mano, etc.

- **Con DABs:** Simplemente abres tu terminal y escribes:

In [0]:
%%bash
databricks bundle deploy -t dev

Databricks leerá tus archivos YAML y creará automáticamente el Job, el Cluster y subirá el código en un solo paso.

### **Databricks CLI**